# 1. IMPORTACIÓN DE DATOS

In [1]:
# Parámetros a definir

    # lesk: True/False
        # True --> Synset se elige utilizando el desambiguador de Lesk
        # False --> Synset es el más común 
    # tipo_bow: --> Categorías de synsets que cogeremos para hacer los cálculos
        # todos (1) --> Todos los synsets
        # nv (2) --> Solo nombres y adjetivos
    # metodo_frase: lw / Ariadna / Agirre
    # partición: numero entero
        # fragmento del corpus que se procesa.
        

desambiguar = True

expandir_contracciones = False

tipo_bow = 'nv'

metodo_frase = 'Agirre'

particion = 1

print_info = False

importar_librerias = True

importar_datos = True

criterios = 'Lesk-' + str(desambiguar) + ', tipo_bow-' + str(tipo_bow) + ', metodo-' + str(metodo_frase) + ', particion-' + str(particion)

print()
print('Desambiguación desambiguar:', desambiguar)
print('Tipo de BoW:', tipo_bow)
print('Método de comparación a nivel de frase:', metodo_frase)
print('Particion de filas a importar:', particion)
print('Importar librerías:', importar_librerias)
print('Importar datos:', importar_datos)

print(criterios)


Desambiguación desambiguar: True
Tipo de BoW: nv
Método de comparación a nivel de frase: Agirre
Particion de filas a importar: 1
Importar librerías: True
Importar datos: True
Lesk-True, tipo_bow-nv, metodo-Agirre, particion-1


In [2]:
# Importación de librerías

if importar_librerias:
    
    import time
    import datetime

    import csv

    import pandas as pd
    pd.options.display.max_rows  # Para mostrar todas las columnas
    pd.set_option('display.max_colwidth', -1)  # Para incluir todo el contenido de una celda, sin truncar contenido.
    pd.set_option('display.max_columns', 500)  # Para incluir todas las columnas (no serán más de 500)
    pd.set_option('display.max_rows', 6000)  # Para incluir todas las columnas (no serán más de 6000)

    import numpy as np

    from nltk.corpus import wordnet as wn

    from nltk.corpus import wordnet_ic
    ic_brown = wordnet_ic.ic('ic-brown.dat')
    ic_semcor = wordnet_ic.ic('ic-semcor.dat')

    from nltk.corpus import genesis
    ic_genesis = wn.ic(genesis, False, 0.0)
    
    if desambiguar == True:
        from nltk.wsd import lesk




In [3]:
# Funciones generales

# función coseno
    # Entrada: dos vectores
    # Salida: Un número real entre [0,1] (coseno entre los dos vectores)

def coseno(v1, v2):
    
    coseno = v1.dot(v2) / np.sqrt(v1.dot(v1) * v2.dot(v2))
    
    return(coseno)

# Ejemplo para comprobar funcionamiento coseno()

print('****** EJEMPLO coseno() ********')   
v1 = np.array([1,2,3])
v2 = np.array([1,0,1])
print('Coseno:', coseno(v1,v1))
print('Coseno:', coseno(v1,v2))
print('****')


****** EJEMPLO coseno() ********
Coseno: 1.0
Coseno: 0.7559289460184544
****


In [4]:
# Importación de los datos del fichero sts-train.csv a un dataframe

start_total = time.time()

if importar_datos:

    file = 'stsbenchmark\sts-train.csv'
    corpus = pd.read_csv(file, sep='\t', error_bad_lines=False, quoting=csv.QUOTE_NONE, usecols=range(7), header=-1)
    corpus.head()
    corpus.columns = ['genre', 'file', 'year', 'id', 'gold_score', 'sent1', 'sent2']


end_total = time.time()

print('TIEMPO TRANSCURRIDO:', end_total-start_total)

corpus.sample(5)

TIEMPO TRANSCURRIDO: 0.03620719909667969


,genre,file,year,id,gold_score,sent1,sent2
5692,main-news,headlines,2016,1134,1.0,Teenager tests negative for Ebola,Texas hospital worker tests positive for Ebola
1063,main-captions,images,2014,87,3.8,A striped cat laying down on a bag of cat litter.,Domestic cat laying on back of cat litter.
4334,main-news,headlines,2013,734,2.0,Hundreds honor student killed in Ohio shooting,Funeral set for Ohio shooting victim
147,main-captions,MSRvid,2012test,215,3.0,A man is walking slowly across a rope bridge.,A boy is walking across a bridge.
5258,main-news,headlines,2015,761,5.0,Yahoo agrees to buy Tumblr for $1.1 billion cash,Yahoo to Buy Tumblr for $1.1 Billion


# 2. ANÁLISIS 1: REVISIÓN DE LOS DATOS BRUTOS (...)

In [5]:
# Eliminación de duplicados

if importar_datos:

    corpus.drop_duplicates(['sent1', 'sent2'], keep='first', inplace=True)

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

TIEMPO TRANSCURRIDO: 0.07588815689086914


In [6]:
# Creación de un corpus c reducido con filas representativas del corpus original

    # Para realizar pre-procesamientos que nos permitan verificar la adecuación de los algoritmos
    # en un tiempo reducido.

particion = particion

c = corpus.iloc[::particion, :].copy()
num_filas = 'Número de filas incluídas en el dataframe: ' + str(c.shape[0])

print(num_filas)

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

c.head(12)

Número de filas incluídas en el dataframe: 5706
TIEMPO TRANSCURRIDO: 0.0872960090637207


,genre,file,year,id,gold_score,sent1,sent2
0,main-captions,MSRvid,2012test,1,5.00,A plane is taking off.,An air plane is taking off.
1,main-captions,MSRvid,2012test,4,3.80,A man is playing a large flute.,A man is playing a flute.
2,main-captions,MSRvid,2012test,5,3.80,A man is spreading shreded cheese on a pizza.,A man is spreading shredded cheese on an uncooked pizza.
3,main-captions,MSRvid,2012test,6,2.60,Three men are playing chess.,Two men are playing chess.
4,main-captions,MSRvid,2012test,9,4.25,A man is playing the cello.,A man seated is playing the cello.
5,main-captions,MSRvid,2012test,11,4.25,Some men are fighting.,Two men are fighting.
6,main-captions,MSRvid,2012test,12,0.50,A man is smoking.,A man is skating.
7,main-captions,MSRvid,2012test,13,1.60,The man is playing the piano.,The man is playing the guitar.
8,main-captions,MSRvid,2012test,14,2.20,A man is playing on a guitar and singing.,A woman is playing an acoustic guitar and singing.
9,main-captions,MSRvid,2012test,16,5.00,A person is throwing a cat on to the ceiling.,A person throws a cat on the ceiling.


# 3. TOKENIZACION, LIMPIEZA Y ANOTACIÓN

In [7]:
# Pre procesamiento
# https://www.linkedin.com/pulse/processing-normalizing-text-data-saurav-ghosh/
# http://xperimentallearning.blogspot.com/2018/05/nlppython.html

# ========================================
# 3. TOKENIZACION, LIMPIEZA Y ANOTACIÓN
# ========================================

import nltk

def expandir_contracciones(texto):
    # specific
    texto = re.sub(r"won\'t", "will not", texto)
    texto = re.sub(r"can\'t", "can not", texto)

    # general
    texto = re.sub(r"n\'t", " not", texto)
    texto = re.sub(r"\'re", " are", texto)
    texto = re.sub(r"\'s", " is", texto)
    texto = re.sub(r"\'d", " would", texto)
    texto = re.sub(r"\'ll", " will", texto)
    texto = re.sub(r"\'t", " not", texto)
    texto = re.sub(r"\'ve", " have", texto)
    texto = re.sub(r"\'m", " am", texto)
    return texto

def tokenizar(sent, print_tokenizar =False):
    
    # tokens que aceptaremos (de nltk.org/book/ch03). Output: cadena de palabras.
   
    if print_tokenizar == True: print('Frase a tokenizar:', sent)
        
    pattern = r'''(?x)       # set flag to allow verbose regexps
        (?:[A-Z]\.)+         # abbreviations, e.g. U.S.A. ?: needs to be added to specify that the parenthesis specify the scope of the disjunction, not the selection o strings to be extracted
       | \w+(?:-\w+)*        # words with optional internal hyphens
       | [\$,\€]?\d+(?:\.\d+)?%?  # currency and percentages, e.g. $12.40, 82%
    '''
    if print_tokenizar == True: print('pattern:', pattern)
    '''
    AMPLIAR.
    - Analizar errores
    - Alguna solución para detectar phrasal verbs?
    '''

    # tokenización. Output: lista
   
    tokens = nltk.regexp_tokenize(sent, pattern, gaps=False)
    
    # limpieza 1: mayúsculas
    
    tokens = [token.lower() for token in tokens]   
    if print_tokenizar == True: print('Tokens:', tokens)
 
    # anotación POS. Output: lista de tuplas
    
    tagged_tokens = nltk.pos_tag(tokens, tagset='universal')
    
    # Convertimos las etiquetas gramaticales en notación Lesk;
        
    for i, tagged_token in enumerate(tagged_tokens):
        if tagged_tokens[i][1] == 'NOUN': tagged_tokens[i] = (tagged_tokens[i][0], 'n')
        if tagged_tokens[i][1] == 'VERB': tagged_tokens[i] = (tagged_tokens[i][0], 'v')
        if tagged_tokens[i][1] == 'ADJ': tagged_tokens[i] = (tagged_tokens[i][0], 'a')
        if tagged_tokens[i][1] == 'ADV': tagged_tokens[i] = (tagged_tokens[i][0], 'r')
    if print_tokenizar == True: print('Tagged_tokens:', tagged_tokens)
    
    # limipieza 2: stopwords - REVISAR si queremos/podemos incluir phrasal verbs, determinantes, pronombres...
            
                # Wordnet no incluye determinantes o preposiciones, de forma que "a" es asociado por Lesk 
                # al Synset('deoxyadenosine_monophosphate.n.01'), lo cual no nos conviene.
                # Por otro lado, Lesk utiliza para desambiguar solo las palabras que aparecen en Wordnet,
                # de modo que limpiar antes o después de Lesk las stopwords no debería afectar al resultado.
   
    if print_tokenizar == True: print('Limpieza de stopwords')
    stopwords = nltk.corpus.stopwords.words('english')
    tagged_stops = [tagged_stop for tagged_stop in tagged_tokens if tagged_stop[0] in stopwords]
    if print_tokenizar == True: print('Stopwords eliminadas:', tagged_stops)
    tokens = [token for token in tokens if token not in stopwords]
    tagged_tokens = [tagged_token for tagged_token in tagged_tokens if tagged_token[0] not in stopwords]
    if print_tokenizar == True: print('Tagged tokens sin stopwords:', tagged_tokens)

    return(tokens, tagged_tokens, tagged_stops)

# Ejemplo para comprobar funcionamiento tokenizar()
   
sent = "I'll always love my sweet baby"

print('****** EJEMPLO tokenizar() ********')
tokens = tokenizar(sent, print_tokenizar=True)
print('****')

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)


****** EJEMPLO tokenizar() ********
Frase a tokenizar: I'll always love my sweet baby
pattern: (?x)       # set flag to allow verbose regexps
        (?:[A-Z]\.)+         # abbreviations, e.g. U.S.A. ?: needs to be added to specify that the parenthesis specify the scope of the disjunction, not the selection o strings to be extracted
       | \w+(?:-\w+)*        # words with optional internal hyphens
       | [\$,\€]?\d+(?:\.\d+)?%?  # currency and percentages, e.g. $12.40, 82%
    
Tokens: ['i', 'll', 'always', 'love', 'my', 'sweet', 'baby']
Tagged_tokens: [('i', 'n'), ('ll', 'v'), ('always', 'r'), ('love', 'v'), ('my', 'PRON'), ('sweet', 'a'), ('baby', 'n')]
Limpieza de stopwords
Stopwords eliminadas: [('i', 'n'), ('ll', 'v'), ('my', 'PRON')]
Tagged tokens sin stopwords: [('always', 'r'), ('love', 'v'), ('sweet', 'a'), ('baby', 'n')]
****
TIEMPO TRANSCURRIDO: 0.24818968772888184


In [8]:
# Tokenización de c

c['tokens1'], c['tokens2'] = '', ''
c['tagged_tokens1'], c['tagged_tokens2'] = '', ''
c['stops1'], c['stops2'] = '', ''

print_tokenizar = False

for i in c.index:
    c.at[i, 'tokens1'] = tokenizar(c.at[i, 'sent1'], print_tokenizar)[0]
    c.at[i, 'tokens2'] = tokenizar(c.at[i, 'sent2'], print_tokenizar)[0]
    c.at[i, 'tagged_tokens1'] = tokenizar(c.at[i, 'sent1'], print_tokenizar)[1]
    c.at[i, 'tagged_tokens2'] = tokenizar(c.at[i, 'sent2'], print_tokenizar)[1]
    c.at[i, 'stops1'] = tokenizar(c.at[i, 'sent1'], print_tokenizar)[2]
    c.at[i, 'stops2'] = tokenizar(c.at[i, 'sent2'], print_tokenizar)[2]

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

c.head()

TIEMPO TRANSCURRIDO: 52.976975202560425


,genre,file,year,id,gold_score,sent1,sent2,tokens1,tokens2,tagged_tokens1,tagged_tokens2,stops1,stops2
0,main-captions,MSRvid,2012test,1,5.00,A plane is taking off.,An air plane is taking off.,"[plane, taking]","[air, plane, taking]","[(plane, n), (taking, v)]","[(air, n), (plane, n), (taking, v)]","[(a, DET), (is, v), (off, PRT)]","[(an, DET), (is, v), (off, PRT)]"
1,main-captions,MSRvid,2012test,4,3.80,A man is playing a large flute.,A man is playing a flute.,"[man, playing, large, flute]","[man, playing, flute]","[(man, n), (playing, v), (large, a), (flute, n)]","[(man, n), (playing, v), (flute, n)]","[(a, DET), (is, v), (a, DET)]","[(a, DET), (is, v), (a, DET)]"
2,main-captions,MSRvid,2012test,5,3.80,A man is spreading shreded cheese on a pizza.,A man is spreading shredded cheese on an uncooked pizza.,"[man, spreading, shreded, cheese, pizza]","[man, spreading, shredded, cheese, uncooked, pizza]","[(man, n), (spreading, v), (shreded, v), (cheese, n), (pizza, n)]","[(man, n), (spreading, v), (shredded, v), (cheese, n), (uncooked, a), (pizza, n)]","[(a, DET), (is, v), (on, ADP), (a, DET)]","[(a, DET), (is, v), (on, ADP), (an, DET)]"
3,main-captions,MSRvid,2012test,6,2.60,Three men are playing chess.,Two men are playing chess.,"[three, men, playing, chess]","[two, men, playing, chess]","[(three, NUM), (men, n), (playing, v), (chess, n)]","[(two, NUM), (men, n), (playing, v), (chess, n)]","[(are, v)]","[(are, v)]"
4,main-captions,MSRvid,2012test,9,4.25,A man is playing the cello.,A man seated is playing the cello.,"[man, playing, cello]","[man, seated, playing, cello]","[(man, n), (playing, v), (cello, n)]","[(man, n), (seated, v), (playing, v), (cello, n)]","[(a, DET), (is, v), (the, DET)]","[(a, DET), (is, v), (the, DET)]"


In [9]:
# ========================================
# 4. ASIGNACIÓN DE SYNSETS DE WORDNET
# ========================================

# Asignación de synsets 
    # Entrada: lista de tuplas con tokens y sus correspondientes etiquetas gramaticales
    # Argumentos: 
         # lesk: Opción de aplicar el algoritmo de Lesk  [Lesk 1986] de la librería wsd para desambiguar
                # Opción por defecto: n (no) --> Se asignará el primer synset del grupo de synsets (el más común)
                # Opción: s (sí) --> Asignar el synset determinado por Lesk
                
def syns(tagged_tokens, desambiguar=desambiguar):
    from nltk.wsd import lesk
    for i in c.index:
        syns = list()
        errors = list()
        if desambiguar == True:
            for j in tagged_tokens:
                if lesk([token for token, tag in tagged_tokens], j[0], j[1]) is not None:
                    syns.append(lesk([token for token, tag in tagged_tokens], j[0], j[1]))
                else:
                    errors.append(j)
        else: 
            for j in tagged_tokens:
                try: syns.append(wn.synsets(j[0],j[1])[0])
                except: errors.append(j)
        return(syns, errors)
    
# Ejemplo para comprobar funcionamiento syns()

desambiguar = True

print('Desambiguación Lesk:', desambiguar)

tagged_tokens = c['tagged_tokens1'][151]
syns1 = syns(tagged_tokens, desambiguar)

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

print('****** EJEMPLO syns() ********')   
print('tagged_tokens:', tagged_tokens)
print('syns(tagged_tokens):', syns1[0])
print('syns(tagged_tokens) - Errors:', syns1[1])
print('****')

desambiguar = False

print('Desambiguación Lesk:', desambiguar)

tagged_tokens = c['tagged_tokens1'][151]
syns1 = syns(tagged_tokens, desambiguar)

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

print('****** EJEMPLO syns() ********')   
print('tagged_tokens:', tagged_tokens)
print('syns(tagged_tokens):', syns1[0])
print('syns(tagged_tokens) - Errors:', syns1[1])
print('****')

Desambiguación Lesk: True
TIEMPO TRANSCURRIDO: 53.02310752868652
****** EJEMPLO syns() ********
tagged_tokens: [('man', 'n'), ('cutting', 'v'), ('fish', 'a')]
syns(tagged_tokens): [Synset('world.n.08'), Synset('cut.v.24')]
syns(tagged_tokens) - Errors: [('fish', 'a')]
****
Desambiguación Lesk: False
TIEMPO TRANSCURRIDO: 53.024067640304565
****** EJEMPLO syns() ********
tagged_tokens: [('man', 'n'), ('cutting', 'v'), ('fish', 'a')]
syns(tagged_tokens): [Synset('man.n.01'), Synset('cut.v.01')]
syns(tagged_tokens) - Errors: [('fish', 'a')]
****


In [10]:
# Asignacion de synsets a los tokens de c

desambiguar = desambiguar
print('Desambiguación Lesk:', desambiguar)
tipo_bow = tipo_bow
print('Tipo_bow:', tipo_bow)

c['syns1'] = ''
c['syns2'] = ''
c['errors1'] = ''
c['errors2'] = ''
c['syns1_nv'] = ''
c['syns2_nv'] = ''
c['syns1_resto'] = ''
c['syns2_resto'] = ''

for i in c.index:
    c.at[i,'syns1'] = syns(c.at[i, 'tagged_tokens1'])[0]
    c.at[i,'syns2'] = syns(c.at[i, 'tagged_tokens2'])[0]
    c.at[i,'errors1'] = syns(c.at[i, 'tagged_tokens1'])[1]
    c.at[i,'errors2'] = syns(c.at[i, 'tagged_tokens2'])[1]
    
    if tipo_bow == 'nv':
        c.at[i,'syns1_nv'] = [syn for syn in c.at[i,'syns1'] if syn.pos() in ['n', 'v']]
        c.at[i,'syns2_nv'] = [syn for syn in c.at[i,'syns2'] if syn.pos() in ['n', 'v']]
        c.at[i,'syns1_resto'] = [syn for syn in c.at[i,'syns1'] if syn.pos() not in ['n', 'v']]
        c.at[i,'syns2_resto'] = [syn for syn in c.at[i,'syns2'] if syn.pos() not in ['n', 'v']]


end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

c.sample(5)

Desambiguación Lesk: False
Tipo_bow: nv
TIEMPO TRANSCURRIDO: 68.31379508972168


,genre,file,year,id,gold_score,sent1,sent2,tokens1,tokens2,tagged_tokens1,tagged_tokens2,stops1,stops2,syns1,syns2,errors1,errors2,syns1_nv,syns2_nv,syns1_resto,syns2_resto
860,main-captions,MSRvid,2012train,537,0.8,A man takes a slice of pizza.,A man talks on his cell phone.,"[man, takes, slice, pizza]","[man, talks, cell, phone]","[(man, n), (takes, v), (slice, n), (pizza, n)]","[(man, n), (talks, n), (cell, n), (phone, n)]","[(a, DET), (a, DET), (of, ADP)]","[(a, DET), (on, ADP), (his, PRON)]","[Synset('world.n.08'), Synset('take.v.41'), Synset('slice.n.06'), Synset('pizza.n.01')]","[Synset('world.n.08'), Synset('talk.n.05'), Synset('cellular_telephone.n.01'), Synset('telephone.n.01')]",[],[],"[Synset('world.n.08'), Synset('take.v.41'), Synset('slice.n.06'), Synset('pizza.n.01')]","[Synset('world.n.08'), Synset('talk.n.05'), Synset('cellular_telephone.n.01'), Synset('telephone.n.01')]",[],[]
3179,main-news,MSRpar,2012train,355,3.0,"Besides Hampton and Newport News, the grant funds water testing in Yorktown, King George County, Norfolk and Virginia Beach.","The grant also funds beach testing in King George County, Norfolk and Virginia Beach.","[besides, hampton, newport, news, grant, funds, water, testing, yorktown, king, george, county, norfolk, virginia, beach]","[grant, also, funds, beach, testing, king, george, county, norfolk, virginia, beach]","[(besides, ADP), (hampton, n), (newport, a), (news, n), (grant, n), (funds, n), (water, n), (testing, v), (yorktown, a), (king, n), (george, n), (county, n), (norfolk, n), (virginia, n), (beach, n)]","[(grant, n), (also, r), (funds, n), (beach, v), (testing, v), (king, v), (george, n), (county, n), (norfolk, n), (virginia, n), (beach, n)]","[(and, CONJ), (the, DET), (in, ADP), (and, CONJ)]","[(the, DET), (in, ADP), (and, CONJ)]","[Synset('hampton.n.01'), Synset('newsworthiness.n.01'), Synset('grant.n.08'), Synset('store.n.02'), Synset('water_system.n.02'), Synset('test.v.07'), Synset('king.n.09'), Synset('george.n.07'), Synset('county.n.02'), Synset('norfolk.n.01'), Synset('virginia.n.03'), Synset('beach.n.01')]","[Synset('grant.n.08'), Synset('besides.r.02'), Synset('store.n.02'), Synset('beach.v.01'), Synset('test.v.07'), Synset('george.n.07'), Synset('county.n.02'), Synset('norfolk.n.01'), Synset('virginia.n.03'), Synset('beach.n.01')]","[(besides, ADP), (newport, a), (yorktown, a)]","[(king, v)]","[Synset('hampton.n.01'), Synset('newsworthiness.n.01'), Synset('grant.n.08'), Synset('store.n.02'), Synset('water_system.n.02'), Synset('test.v.07'), Synset('king.n.09'), Synset('george.n.07'), Synset('county.n.02'), Synset('norfolk.n.01'), Synset('virginia.n.03'), Synset('beach.n.01')]","[Synset('grant.n.08'), Synset('store.n.02'), Synset('beach.v.01'), Synset('test.v.07'), Synset('george.n.07'), Synset('county.n.02'), Synset('norfolk.n.01'), Synset('virginia.n.03'), Synset('beach.n.01')]",[],[Synset('besides.r.02')]
1120,main-captions,images,2014,169,1.4,People sitting on the porch.,People sitting on acouch.,"[people, sitting, porch]","[people, sitting, acouch]","[(people, n), (sitting, v), (porch, n)]","[(people, n), (sitting, v), (acouch, a)]","[(on, ADP), (the, DET)]","[(on, ADP)]","[Synset('multitude.n.03'), Synset('sit_down.v.01'), Synset('porch.n.01')]","[Synset('multitude.n.03'), Synset('sit_down.v.01')]",[],"[(acouch, a)]","[Synset('multitude.n.03'), Synset('sit_down.v.01'), Synset('porch.n.01')]","[Synset('multitude.n.03'), Synset('sit_down.v.01')]",[],[]
3104,main-news,MSRpar,2012train,234,2.5,"The Dow Jones industrial average .DJI edged up 13.33 points, or 0.15 percent, to 9,196.55.","The Dow Jones industrial average <.DJI> was off 7.75 points, or 0.08 percent, at 9,175.47.","[dow, jones, industrial, average, dji, edged, 13, 33, points, 0, 15, percent, 9, ,196.55]","[dow, jones, industrial, average, dji, 7, 75, points, 0, 08, percent, 9, ,175.47]","[(dow, n), (jones, n), (industrial, a), (average, a), (dji, n), (edged, v), (13, NUM), (33, NUM), (points, n), (0, NUM), (

# 4. ANÁLISIS 2: TIPO Y DISTRIBUCIÓN DE TOKENS Y SYNSETS (...)


# 7. SIMILITUD A NIVEL DE FRASE - SOLAPAMIENTO DE AGIRRE

In [11]:
# 7. SIMILITUD A NIVEL DE FRASE - SOLAPAMIENTO DE AGIRRE

def similitud_frase_agirre(tokens1, tokens2):
    len1 = len(tokens1)
    len2 = len(tokens2)
    alineados = len([word for word in tokens1 if word in tokens2])
    similitud = 2 * alineados / (len1 + len2)
    return(similitud)
    
# Ejemplo para comprobar funcionamiento similitud_frase_agirre()

tokens1 = c['tokens1'][501]
tokens2 = c['tokens2'][501]


print('****** EJEMPLO similitud_frase_agirre() ********')   

print(tokens1)
print(tokens2)
print(similitud_frase_agirre(tokens1, tokens2))

****** EJEMPLO similitud_frase_agirre() ********
['woman', 'cutting', 'onion']
['woman', 'cutting', 'onion']
1.0


In [12]:
# Cálculo y almacenamiento de las similitudes de Agirre

if metodo_frase == 'Agirre':

    c['puntuacion_Agirre'] = 0.0

    for i in c.index:

        start = time.time()

        tokens1 = c.at[i, 'tokens1']
        tokens2 = c.at[i, 'tokens2']

        c.at[i, 'puntuacion_Agirre'] = similitud_frase_agirre(tokens1, tokens2) 

        end = time.time()
        print(end-start)


0.0004949569702148438
0.0004961490631103516
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0004966259002685547
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0004963874816894531
0.0
0.0
0.0004956722259521484
0.0
0.0
0.0
0.0
0.0
0.0004973411560058594
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0004379749298095703
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0004954338073730469
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0004947185516357422
0.0
0.0
0.0
0.0
0.0
0.00049591064453125
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0004975795745849609
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0005009174346923828
0.0
0.0
0.0
0.0
0.0
0.000492095947265625
0.0
0.0004787445068359375
0.0
0.00046634674072265625
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0004975795745849609
0.0004975795745849609
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0004961490631103516
0.0
0.0
0.0
0.0
0.0
0.

0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0004961490631103516
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.00049591064453125
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.00049591064453125
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0004968643188476562
0.0
0.0
0.0
0.0
0.0
0.0
0.0004956722259521484
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0004949569702148438
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0004947185516357422
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0004966259002685547
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0004954338073730469
0.0
0.0
0.0
0.

0.0
0.0
0.0
0.0004949569702148438
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0004944801330566406
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0004956722259521484
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0004949569702148438
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0004971027374267578
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0004956722259521484
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0004968643188476562
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0004968643188476562
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0004956722259521484
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0004961490631103516
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.00049591064453125
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0


In [13]:
x = datetime.datetime.now()
c.to_excel(x.strftime("%y%m%d-%H%M-") + criterios + '.xlsx')

end_total = time.time()

print('TIEMPO TRANSCURRIDO:', end_total-start_total)

TIEMPO TRANSCURRIDO: 73.75785541534424
